In [1]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

from torchvision.models import resnet50, ResNet50_Weights
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
count = sum(p.numel() for p in model.parameters())
print(f"ResNet50 loaded: {count:,} numbers")

PyTorch: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 221MB/s]


ResNet50 loaded: 25,557,032 numbers


In [2]:
import random
from torchvision import datasets
from torch.utils.data import Subset, DataLoader

device = "cuda"
weights = ResNet50_Weights.IMAGENET1K_V2
preprocess = weights.transforms()      # resize, crop to 224x224, scale colours the way ResNet50 expects

# Imagenette: 10 everyday ImageNet categories, downloaded automatically (about 340 MB)
val_set = datasets.Imagenette("data", split="val", size="320px", download=True, transform=preprocess)

# Imagenette labels its classes 0-9; ResNet50 numbers them among its 1,000 ImageNet classes.
IMAGENET_INDEX = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]

# Use a fixed random sample of 1,000 photos, so each accuracy check takes about a second.
N_IMAGES = 1000
random.seed(0)
picked = random.sample(range(len(val_set)), N_IMAGES)
loader = DataLoader(Subset(val_set, picked), batch_size=100, num_workers=2)

image_batches, label_batches = [], []
for x, y in loader:
    image_batches.append(x)
    label_batches.append(y)
images = torch.cat(image_batches).to(device)
labels = torch.tensor([IMAGENET_INDEX[int(y)] for y in torch.cat(label_batches)]).to(device)

model = model.to(device).eval()


def accuracy(model, images, labels, batch=250):
    """Share of the photos the model labels correctly (out of all 1,000 ImageNet classes)."""
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch):
            predictions = model(images[i:i + batch]).argmax(dim=1)
            correct += (predictions == labels[i:i + batch]).sum().item()
    return correct / len(images)


print("check the label mapping:")
for name, index in zip(val_set.classes, IMAGENET_INDEX):
    print(f"  {name}  ->  {weights.meta['categories'][index]}")
print(f"\nphotos: {len(images)}   baseline accuracy: {accuracy(model, images, labels):.2%}")

100%|██████████| 342M/342M [00:21<00:00, 16.1MB/s]


check the label mapping:
  ('tench', 'Tinca tinca')  ->  tench
  ('English springer', 'English springer spaniel')  ->  English springer
  ('cassette player',)  ->  cassette player
  ('chain saw', 'chainsaw')  ->  chain saw
  ('church', 'church building')  ->  church
  ('French horn', 'horn')  ->  French horn
  ('garbage truck', 'dustcart')  ->  garbage truck
  ('gas pump', 'gasoline pump', 'petrol pump', 'island dispenser')  ->  gas pump
  ('golf ball',)  ->  golf ball
  ('parachute', 'chute')  ->  parachute

photos: 1000   baseline accuracy: 89.10%


In [3]:
import copy
import csv

ATTEMPTS = 300
CRITICAL_DROP = 0.10
RESULTS_PATH = "resnet50_sweep.csv"

# name: (float type, same-size integer type, exponent's top bit)
FORMATS = {
    "fp32": (torch.float32, torch.int32, 30),
    "fp16": (torch.float16, torch.int16, 14),
}


def all_numbers(model):
    """Every weight and bias tensor, flattened into one long row each."""
    return [(name, p.view(-1)) for name, p in model.named_parameters()]


def pick_random_number(params, rng):
    """Pick one of the model's 25.6 million numbers uniformly at random."""
    k = rng.randrange(sum(flat.numel() for _, flat in params))
    for name, flat in params:
        if k < flat.numel():
            return name, flat, k
        k -= flat.numel()


def flip_in_memory(flat, index, int_type, bit):
    """Flip one bit of flat[index] directly in memory. Calling it twice undoes it."""
    as_int = flat.view(int_type)
    as_int[index] ^= (1 << bit)


def layer_kind(name):
    if name.startswith("fc"):
        return "fc (final layer)"
    if "bn" in name or "downsample.1" in name:
        return "batchnorm"
    return "conv"


def sweep(fmt):
    float_type, int_type, bit = FORMATS[fmt]
    m = copy.deepcopy(model).to(float_type)       # a separate copy in this format
    x = images.to(float_type)
    baseline = accuracy(m, x, labels)
    rng = random.Random(0)                         # same seed -> same picks in both formats
    params = all_numbers(m)
    rows = []
    for attempt in range(1, ATTEMPTS + 1):
        layer, flat, index = pick_random_number(params, rng)
        before = flat[index].item()
        with torch.no_grad():
            flip_in_memory(flat, index, int_type, bit)    # simulated Rowhammer flip
            after = flat[index].item()
            acc = accuracy(m, x, labels)
            flip_in_memory(flat, index, int_type, bit)    # flip again = restore
        rows.append({"format": fmt, "attempt": attempt, "layer": layer,
                     "kind": layer_kind(layer), "index": index,
                     "value_before": before, "value_after": after,
                     "accuracy": acc, "critical": baseline - acc > CRITICAL_DROP})
        if attempt % 50 == 0:
            print(f"  {fmt}: {attempt}/{ATTEMPTS} done")
    del m
    torch.cuda.empty_cache()
    return baseline, rows


all_rows = []
for fmt in FORMATS:
    baseline, rows = sweep(fmt)
    all_rows += rows
    crit = [r for r in rows if r["critical"]]
    rate = len(crit) / len(rows)
    worst = min(rows, key=lambda r: r["accuracy"])
    print(f"\n{fmt}: baseline {baseline:.2%}, critical {len(crit)}/{len(rows)} = {rate:.1%}, "
          f"expected attempts {1 / rate:.1f}" if rate else f"\n{fmt}: no critical attempts")
    print(f"  worst: {worst['layer']} {worst['value_before']:.4g} -> {worst['value_after']:.4g}, "
          f"accuracy {worst['accuracy']:.2%}")
    for kind in sorted({r["kind"] for r in rows}):
        hits = [r for r in rows if r["kind"] == kind]
        bad = sum(r["critical"] for r in hits)
        print(f"  {kind:<17} attempts {len(hits):>4}   critical {bad:>4}")

with open(RESULTS_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=all_rows[0].keys())
    writer.writeheader()
    writer.writerows(all_rows)
print(f"\nsaved {RESULTS_PATH}")

  fp32: 50/300 done
  fp32: 100/300 done
  fp32: 150/300 done
  fp32: 200/300 done
  fp32: 250/300 done
  fp32: 300/300 done

fp32: baseline 89.10%, critical 193/300 = 64.3%, expected attempts 1.6
  worst: layer4.1.conv2.weight 0.01856 -> 6.316e+36, accuracy 0.00%
  batchnorm         attempts    1   critical    0
  conv              attempts  276   critical  184
  fc (final layer)  attempts   23   critical    9
  fp16: 50/300 done
  fp16: 100/300 done
  fp16: 150/300 done
  fp16: 200/300 done
  fp16: 250/300 done
  fp16: 300/300 done

fp16: baseline 89.20%, critical 149/300 = 49.7%, expected attempts 2.0
  worst: layer2.3.conv2.weight 0.02232 -> 1463, accuracy 0.00%
  batchnorm         attempts    1   critical    0
  conv              attempts  276   critical  141
  fc (final layer)  attempts   23   critical    8

saved resnet50_sweep.csv


In [4]:
for fmt in FORMATS:
    print(f"\n{fmt}: conv-layer flips, by sign of the original weight")
    conv = [r for r in all_rows if r["format"] == fmt and r["kind"] == "conv"]
    for label, keep in (("positive", lambda v: v > 0), ("negative", lambda v: v < 0)):
        group = [r for r in conv if keep(r["value_before"])]
        bad = sum(r["critical"] for r in group)
        print(f"  {label}: {bad} / {len(group)} = {bad / len(group):.0%}")


fp32: conv-layer flips, by sign of the original weight
  positive: 143 / 143 = 100%
  negative: 41 / 133 = 31%

fp16: conv-layer flips, by sign of the original weight
  positive: 141 / 143 = 99%
  negative: 0 / 133 = 0%


In [5]:
check_model = copy.deepcopy(model).float()
params = dict(check_model.named_parameters())
x = images[:250]

neg_fp32 = [r for r in all_rows
            if r["format"] == "fp32" and r["kind"] == "conv" and r["value_before"] < 0]
critical = [r for r in neg_fp32 if r["critical"]][:10]
harmless = [r for r in neg_fp32 if not r["critical"]][:10]


def output_has_nan(r):
    """Re-do one flip, check whether the model's outputs contain NaN, then undo it."""
    flat = params[r["layer"]].view(-1)
    with torch.no_grad():
        flip_in_memory(flat, r["index"], torch.int32, 30)
        out = check_model(x)
        flip_in_memory(flat, r["index"], torch.int32, 30)
    return torch.isnan(out).any().item()


for label, group in (("critical negative flips", critical), ("harmless negative flips", harmless)):
    nan_count = sum(output_has_nan(r) for r in group)
    print(f"{label}: {nan_count} / {len(group)} produced NaN outputs")

critical negative flips: 3 / 10 produced NaN outputs
harmless negative flips: 0 / 10 produced NaN outputs


In [6]:
from google.colab import files
files.download("resnet50_sweep.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>